In [ ]:
# Install arc-agi from competition wheels
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv 2>&1 | tail -5

In [ ]:
# CRITICAL: Create submission.parquet for notebook validation
import pandas as pd, os
_df = pd.DataFrame([{'row_id':'1_0','game_id':'1','end_of_game':True,'score':1.0}])
_df.to_parquet('/kaggle/working/submission.parquet', index=False)
print('submission.parquet created')

In [ ]:
%%writefile /kaggle/working/my_agent.py
# HIRA-X v15 - Hybrid Intelligence & Reasoning Agent-X
import hashlib, logging, os, random, time, traceback
from collections import defaultdict, deque
from typing import Any, Dict, List, Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState

class Config:
    GRID_SIZE = 64
    NUM_COLOURS = 16
    NUM_ACTION_TYPES = 5
    TOTAL_ACTIONS = NUM_ACTION_TYPES + GRID_SIZE * GRID_SIZE
    LR = 3e-4
    BATCH_SIZE = 64
    ENTROPY_WEIGHT = 0.01
    GRAD_CLIP = 1.0
    BUFFER_SIZE = 150_000
    TRAIN_FREQ = 4
    MCTS_SIMULATIONS = 30

class GridAttentionEncoder(nn.Module):
    def __init__(self, input_channels=16, embed_dim=128, num_heads=4):
        super().__init__()
        self.patch_embed = nn.Conv2d(input_channels, embed_dim, kernel_size=8, stride=8)
        num_patches = (64 // 8) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*4, dropout=0.1, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.pool = nn.AdaptiveAvgPool1d(1)
    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        x = self.transformer(x).transpose(1, 2)
        return self.pool(x).squeeze(-1)

class HybridActionNet(nn.Module):
    def __init__(self, input_channels=16, grid_size=64):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.transformer = GridAttentionEncoder(input_channels, embed_dim=128)
        self.cnn_pool = nn.AdaptiveAvgPool2d(16)
        self.action_head = nn.Sequential(nn.Linear(128*16*16 + 128, 512), nn.ReLU(), nn.Dropout(0.2), nn.Linear(512, 5))
        self.coord_upsample = nn.Sequential(nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.Conv2d(32, 1, 1))
    def forward(self, x):
        c1 = F.relu(self.bn1(self.conv1(x)))
        c2 = F.relu(self.bn2(self.conv2(c1)))
        c3 = F.relu(self.bn3(self.conv3(c2)))
        t_feat = self.transformer(x)
        c_pooled = self.cnn_pool(c3).flatten(1)
        action_logits = self.action_head(torch.cat([c_pooled, t_feat], dim=1))
        coord_map = self.coord_upsample(c3).view(c3.size(0), -1)
        return action_logits, coord_map

class MCTSNode:
    def __init__(self, state_hash, parent=None, action=None):
        self.state_hash = state_hash
        self.parent = parent
        self.action = action
        self.children = {}
        self.visits = 0
        self.value_sum = 0.0
        self.prior = 1.0
    @property
    def value(self):
        return float('inf') if self.visits == 0 else self.value_sum / self.visits
    def ucb_score(self, c=1.414):
        return float('inf') if self.visits == 0 else self.value + c * self.prior * (self.parent.visits**0.5 / (self.visits+1e-8))

class SimpleMCTS:
    def __init__(self, model, device, num_simulations=30):
        self.model = model
        self.device = device
        self.num_simulations = num_simulations
    def search(self, state_tensor, available_mask, temperature=1.0):
        root = MCTSNode("root")
        for _ in range(self.num_simulations):
            node = root
            path = [node]
            while node.children:
                node = max(node.children.values(), key=lambda n: n.ucb_score())
                path.append(node)
            if node.children:
                action = np.random.choice(list(node.children.keys()))
                node = node.children[action]
                path.append(node)
            value = 0.5
            for n in reversed(path):
                n.visits += 1
                n.value_sum += value
        counts = np.zeros(len(available_mask))
        for action, child in root.children.items():
            counts[action] = child.visits
        if temperature == 0:
            probs = np.zeros_like(counts)
            probs[np.argmax(counts)] = 1.0
        else:
            counts_temp = counts ** (1.0 / temperature)
            probs = counts_temp / (counts_temp.sum() + 1e-8)
        return probs

class StateGraph:
    def __init__(self, num_action_types, grid_size):
        self.total_actions = num_action_types + grid_size * grid_size
        self.edges = {}
        self.adjacency = defaultdict(list)
        self.visit_count = defaultdict(int)
        self.tried_actions = defaultdict(set)
        self.current_state = None
        self.pending_action = None
        self.all_states = {}
    @staticmethod
    def _hash_frame(frame_np):
        return hashlib.md5(frame_np.tobytes()).hexdigest()
    def observe(self, frame_np):
        h = self._hash_frame(frame_np)
        if h not in self.all_states:
            self.all_states[h] = frame_np
        self.visit_count[h] += 1
        if self.current_state is not None and self.pending_action is not None:
            key = (self.current_state, self.pending_action)
            if key not in self.edges:
                self.edges[key] = h
                self.adjacency[self.current_state].append((self.pending_action, h))
                self.tried_actions[self.current_state].add(self.pending_action)
        self.current_state = h
        self.pending_action = None
        return h
    def record_action(self, action_idx):
        self.pending_action = action_idx
    def untried_actions(self, state_hash, available_mask):
        tried = self.tried_actions.get(state_hash, set())
        candidates = np.where(available_mask)[0] if available_mask is not None else np.arange(self.total_actions)
        return [int(a) for a in candidates if a not in tried]
    def bfs_to_frontier(self, start_hash, available_mask):
        if self.untried_actions(start_hash, available_mask):
            return []
        queue = deque([(start_hash, [])])
        visited = {start_hash}
        while queue:
            node, path = queue.popleft()
            if len(path) > 15:
                break
            for action_idx, nxt in self.adjacency.get(node, []):
                if nxt in visited:
                    continue
                new_path = path + [action_idx]
                if self.untried_actions(nxt, available_mask):
                    return new_path
                visited.add(nxt)
                queue.append((nxt, new_path))
        return None
    def reset_level(self):
        self.edges.clear()
        self.adjacency.clear()
        self.visit_count.clear()
        self.tried_actions.clear()
        self.current_state = None
        self.pending_action = None
        self.all_states.clear()

class DeduplicatedBuffer:
    def __init__(self, maxlen=150_000):
        self.maxlen = maxlen
        self.buffer = []
        self.seen = set()
        self.write_pos = 0
        self.size = 0
    def add(self, state_np, action_idx, reward):
        state_hash = hashlib.md5(state_np.tobytes()).hexdigest()
        key = (state_hash, action_idx)
        if key in self.seen:
            return
        self.seen.add(key)
        entry = {"state": state_np, "action_idx": action_idx, "reward": reward, "_key": key}
        if self.size < self.maxlen:
            self.buffer.append(entry)
            self.size += 1
        else:
            old = self.buffer[self.write_pos]
            if old:
                self.seen.discard(old["_key"])
            self.buffer[self.write_pos] = entry
        self.write_pos = (self.write_pos + 1) % self.maxlen
    def sample(self, batch_size):
        if self.size <= batch_size:
            return self.buffer[:self.size]
        indices = np.random.choice(self.size, batch_size, replace=False)
        return [self.buffer[i] for i in indices]
    def __len__(self):
        return self.size
    def clear(self):
        self.buffer.clear()
        self.seen.clear()
        self.write_pos = 0
        self.size = 0

class MyAgent(Agent):
    MAX_ACTIONS = float('inf')
    _MAX_FRAMES = 10
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        seed = int(time.time()*1000000) + hash(self.game_id)%1000000
        random.seed(seed)
        np.random.seed(seed % (2**32-1))
        torch.manual_seed(seed % (2**32-1))
        self.start_time = time.time()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"[{self.game_id}] HIRA-X v15 on {self.device}")
        self.logger = logging.getLogger(f"HIRA_{self.game_id}")
        self.graph = StateGraph(Config.NUM_ACTION_TYPES, Config.GRID_SIZE)
        self.model = HybridActionNet(Config.NUM_COLOURS, Config.GRID_SIZE).to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=Config.LR)
        self.mcts = SimpleMCTS(self.model, self.device, num_simulations=30)
        self.buffer = DeduplicatedBuffer(maxlen=Config.BUFFER_SIZE)
        self.action_list = [GameAction.ACTION1, GameAction.ACTION2, GameAction.ACTION3, GameAction.ACTION4, GameAction.ACTION5]
        self.current_level = -1
        self.planned_path = []
        self.prev_frame_np = None
        self.prev_action_idx = None
        self.step_count = 0
    def _frame_to_np(self, frame_data):
        try:
            arr = np.array(frame_data.frame, dtype=np.uint8)
            frame = arr[-1]
            one_hot = np.zeros((Config.NUM_COLOURS, Config.GRID_SIZE, Config.GRID_SIZE), dtype=np.float32)
            for c in range(Config.NUM_COLOURS):
                one_hot[c] = (frame == c).astype(np.float32)
            return one_hot
        except:
            return None
    def _np_to_tensor(self, frame_np):
        return torch.from_numpy(frame_np).unsqueeze(0).to(self.device)
    def _build_mask(self, available_actions):
        mask = np.zeros(Config.TOTAL_ACTIONS, dtype=bool)
        has_coord = False
        if available_actions:
            for act in available_actions:
                aid = act.value if hasattr(act, 'value') else int(act)
                if 1 <= aid <= 5:
                    mask[aid-1] = True
                elif aid == 6:
                    has_coord = True
        else:
            mask[:5] = True
            has_coord = True
        if has_coord:
            mask[5:] = True
        return mask
    def _saliency(self, frame_np):
        self.model.eval()
        t = self._np_to_tensor(frame_np)
        with torch.no_grad():
            act_logits, coord_logits = self.model(t)
        self.model.train()
        act_probs = torch.sigmoid(act_logits).squeeze(0).cpu().numpy()
        coord_probs = torch.sigmoid(coord_logits).squeeze(0).cpu().numpy()
        return np.concatenate([act_probs, coord_probs])
    def _action_idx_to_game(self, idx, available_actions):
        if idx < 5:
            act = self.action_list[idx]
            act.reasoning = f"{act.name} v15"
            return act
        coord_idx = idx - 5
        y = coord_idx // Config.GRID_SIZE
        x = coord_idx % Config.GRID_SIZE
        act = GameAction.ACTION6
        act.set_data({"x": int(x), "y": int(y)})
        act.reasoning = f"A6({x},{y}) v15"
        return act
    def _train(self):
        if len(self.buffer) < Config.BATCH_SIZE:
            return
        batch = self.buffer.sample(Config.BATCH_SIZE)
        states = torch.stack([torch.from_numpy(e["state"]).float().to(self.device) for e in batch])
        actions = torch.tensor([e["action_idx"] for e in batch], dtype=torch.long, device=self.device)
        rewards = torch.tensor([e["reward"] for e in batch], dtype=torch.float32, device=self.device)
        act_logits, coord_logits = self.model(states)
        logits = torch.cat([act_logits, coord_logits], dim=1)
        chosen = logits.gather(1, actions.unsqueeze(1)).squeeze(1)
        loss = F.binary_cross_entropy_with_logits(chosen, rewards)
        probs = torch.sigmoid(logits)
        entropy = -(probs * torch.log(probs + 1e-8)).mean()
        loss = loss - Config.ENTROPY_WEIGHT * entropy
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.model.parameters(), Config.GRAD_CLIP)
        self.optimizer.step()
    def _reset_level(self, new_level):
        self.graph.reset_level()
        self.planned_path = []
        self.prev_frame_np = None
        self.prev_action_idx = None
        self.buffer.clear()
        self.current_level = new_level
    def append_frame(self, frame):
        self.frames.append(frame)
        if len(self.frames) > self._MAX_FRAMES:
            self.frames = self.frames[-self._MAX_FRAMES:]
        if hasattr(self, "recorder") and not self.is_playback:
            import json
            self.recorder.record(json.loads(frame.model_dump_json()))
    def is_done(self, frames, latest):
        try:
            return latest.state is GameState.WIN or (time.time()-self.start_time) >= 8*3600-300
        except:
            return True
    def choose_action(self, frames, latest):
        try:
            return self._choose_impl(latest)
        except Exception as e:
            traceback.print_exc()
            act = random.choice(self.action_list[:3])
            act.reasoning = f"Crash: {e}"
            return act
    def _choose_impl(self, latest):
        level = getattr(latest, "levels_completed", 0)
        if level != self.current_level:
            self._reset_level(level)
        if latest.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            self.graph.reset_level()
            self.planned_path = []
            act = GameAction.RESET
            act.reasoning = "Reset"
            return act
        frame_np = self._frame_to_np(latest)
        if frame_np is None:
            act = random.choice(self.action_list[:3])
            act.reasoning = "Bad frame"
            return act
        state_hash = self.graph.observe(frame_np)
        if self.prev_frame_np is not None and self.prev_action_idx is not None:
            reward = 1.0 if state_hash != hashlib.md5(self.prev_frame_np.tobytes()).hexdigest() else 0.0
            self.buffer.add(self.prev_frame_np, self.prev_action_idx, reward)
        available = getattr(latest, "available_actions", None)
        mask = self._build_mask(available)
        if self.planned_path:
            action_idx = self.planned_path.pop(0)
            if mask[action_idx]:
                self.graph.record_action(action_idx)
                self.prev_frame_np = frame_np
                self.prev_action_idx = action_idx
                return self._action_idx_to_game(action_idx, available)
            self.planned_path = []
        untried = self.graph.untried_actions(state_hash, mask)
        if untried:
            state_tensor = self._np_to_tensor(frame_np)
            mcts_probs = self.mcts.search(state_tensor, mask, temperature=1.0)
            untried_probs = mcts_probs * np.array([1 if i in untried else 0 for i in range(len(mcts_probs))])
            if untried_probs.sum() > 0:
                untried_probs /= untried_probs.sum()
                action_idx = np.random.choice(len(mcts_probs), p=untried_probs)
            else:
                action_idx = random.choice(untried)
        else:
            path = self.graph.bfs_to_frontier(state_hash, mask)
            if path:
                self.planned_path = path[1:] if len(path) > 1 else []
                action_idx = path[0]
            else:
                sal = self._saliency(frame_np)
                sal[~mask] = -np.inf
                action_idx = int(np.argmax(sal))
        self.graph.record_action(action_idx)
        self.prev_frame_np = frame_np
        self.prev_action_idx = action_idx
        self.step_count += 1
        if self.step_count % Config.TRAIN_FREQ == 0:
            self._train()
        return self._action_idx_to_game(action_idx, available)


In [ ]:
import os, subprocess
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print('=== COMPETITION MODE ===')
    for i in range(60):
        r = subprocess.run(['curl', '-sf', 'http://gateway:8001/api/games'], capture_output=True, timeout=10)
        if r.returncode == 0:
            print('Gateway ready')
            break
        import time; time.sleep(5)
    WORK = '/kaggle/working/ARC-AGI-3-Agents'
    SRC = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents'
    subprocess.run(['cp', '-r', SRC, WORK], check=True)
    subprocess.run(['cp', '/kaggle/working/my_agent.py', f'{WORK}/agents/templates/my_agent.py'], check=True)
    init_py = '''from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent
load_dotenv()
AVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}
'''
    with open(f'{WORK}/agents/__init__.py', 'w') as f:
        f.write(init_py)
    env_content = 'SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=***\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE=online\nRECORDINGS_DIR=/kaggle/working/server_recording\n'
    with open(f'{WORK}/.env', 'w') as f:
        f.write(env_content)
    os.chdir(WORK)
    os.environ['MPLBACKEND'] = 'agg'
    result = subprocess.run(['python', 'main.py', '--agent', 'myagent'], capture_output=True, text=True)
    print(f'Exit code: {result.returncode}')
    if result.stdout:
        print('STDOUT:', result.stdout[-1000:])
    if result.stderr:
        print('STDERR:', result.stderr[-500:])
else:
    print('=== NOT COMPETITION MODE ===')
